# 01 · Corpus de falacias: FallacyES y FallacyES-Political

Primer notebook del preprocesado (E2). Aquí se descargan y se exploran los tres corpus de falacias en español que usa el proyecto:

| Corpus | Qué contiene | Fuente |
|---|---|---|
| FallacyES — prototípicas | Falacias del corpus *Logic* traducidas al español (DeepL + revisión manual) y, para muchas, una versión sin falacia escrita a mano | idUS (Univ. de Sevilla), [DOI 10.12795/11441/167050](https://doi.org/10.12795/11441/167050) |
| FallacyES — espontáneas | Fragmentos de comentarios de meneame.net, con y sin falacia | idUS (Univ. de Sevilla), [DOI 10.12795/11441/167050](https://doi.org/10.12795/11441/167050) |
| FallacyES-Political | Falacias de 19 debates electorales españoles (1993–2023), transcritos con WhisperX | Zenodo, [DOI 10.5281/zenodo.14836328](https://doi.org/10.5281/zenodo.14836328) |

Los tres tienen licencia CC BY 4.0, así que hay que citarlos:
- Cruz, F. L., Troyano, J. A., Enríquez, F. y Ortega, F. J. (2023). Detección y clasificación de falacias prototípicas y espontáneas en español. *Procesamiento del Lenguaje Natural*, 71, 53–62.
- Cruz, F. L., Enríquez, F., Ortega, F. J. y Troyano, J. A. (2025). FallacyES-Political: A multiclass dataset of fallacies in Spanish political debates. *Procesamiento del Lenguaje Natural*, 74, 127–138.

**Por qué estos corpus y no *logical-fallacy* (el original en inglés):** ya están en español, así que no hay que traducir; traen ejemplos sin falacia, que *logical-fallacy* no tiene; y FallacyES-Political es del mismo dominio que el proyecto (discurso político español hablado).

## 0. Preparación

In [1]:
from pathlib import Path

import pandas as pd
import requests

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_FALLACYES = Path("../data/raw/fallacyes")
CARPETA_POLITICAL = Path("../data/raw/fallacyes_political")

# Para ver los textos (casi) enteros al mostrar tablas.
pd.set_option("display.max_colwidth", 200)


def contar_palabras(textos):
    """Número de palabras de cada texto (separando por espacios)."""
    return textos.str.split().str.len()

## 1. Descarga

Los ficheros se guardan en `data/raw/`, que no se sube a GitHub: cada miembro del grupo los descarga ejecutando esta sección. Las direcciones apuntan a versiones fijas (el registro v1.0.0 de Zenodo y ficheros concretos de idUS), así que todos trabajamos con los mismos datos.

Además de los datos se bajan la documentación de FallacyES (`FallacyES_readme.txt`), la guía de anotación de FallacyES-Political (`guia-anotacion.md`) y la lista de debates (`debates_info.csv`).

In [2]:
# Dónde se guarda cada fichero → de dónde se descarga.
DESCARGAS = {
    CARPETA_FALLACYES / "FallacyES-prototypical.csv":
        "https://idus.us.es/server/api/core/bitstreams/1a365d67-2543-47a4-9be3-e7b8ec2cec8f/content",
    CARPETA_FALLACYES / "FallacyES-spontaneous.csv":
        "https://idus.us.es/server/api/core/bitstreams/71a0a09c-f559-4637-b008-35ad5a6e765e/content",
    CARPETA_FALLACYES / "FallacyES_readme.txt":
        "https://idus.us.es/server/api/core/bitstreams/7878d236-7122-4749-9669-64da71a23e08/content",
    CARPETA_POLITICAL / "fallacyES-political.csv":
        "https://zenodo.org/records/14836328/files/fallacyES-political.csv?download=1",
    CARPETA_POLITICAL / "debates_info.csv":
        "https://zenodo.org/records/14836328/files/debates_info.csv?download=1",
    CARPETA_POLITICAL / "guia-anotacion.md":
        "https://zenodo.org/records/14836328/files/guia-anotacion.md?download=1",
}


def descargar(url, destino):
    """Descarga un fichero, salvo que ya exista."""
    if destino.exists():
        print(f"Ya existe: {destino.name}")
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()  # se para con un error si la descarga falla
    destino.write_bytes(respuesta.content)
    print(f"Descargado: {destino.name} ({len(respuesta.content):,} bytes)")


for destino, url in DESCARGAS.items():
    descargar(url, destino)

Ya existe: FallacyES-prototypical.csv
Ya existe: FallacyES-spontaneous.csv
Ya existe: FallacyES_readme.txt
Ya existe: fallacyES-political.csv
Ya existe: debates_info.csv
Ya existe: guia-anotacion.md


## 2. FallacyES — falacias prototípicas

Cada fila es una falacia (`fallacy_example`) con su tipo (`label`) y, cuando los autores la escribieron, una versión corregida sin falacia (`non-fallacy_example`). Esas versiones sin falacia se crearon a mano modificando la falacia, así que no son textos reales.

In [3]:
proto = pd.read_csv(CARPETA_FALLACYES / "FallacyES-prototypical.csv")
print(proto.shape)
proto.head()

(2083, 3)


,label,fallacy_example,non-fallacy_example
0,faulty generalization,"""A Annie le debe gustar Starbucks porque a todas las chicas blancas les gusta Starbucks"".",A Annie le debe gustar Starbucks porque a ella le gustan las bebidas dulces y Starbucks tiene muchas opciones de bebidas dulces.
1,faulty generalization,"Este año hace más calor en Las Vegas que el año pasado; por lo tanto, el calentamiento global se está acelerando rápidamente.","El año pasado hacía más calor en Las Vegas que este año; por lo tanto, el calentamiento global se está ralentizando."
2,faulty generalization,"""Los dos cursos que tomé en la UF no fueron muy interesantes. No creo que sea una buena universidad"".",Los dos cursos que tomé en la UF me parecieron aburridos. No tengo una opinión formada sobre si la UF es una buena universidad o no.
3,faulty generalization,Un conductor con matrícula de Nueva York te corta el paso en el tráfico. Decides que todos los conductores de Nueva York son pésimos conductores.,Un conductor con matrícula de Nueva York te corta el paso en el tráfico. Esto no significa que todos los conductores de Nueva York sean pésimos conductores.
4,faulty generalization,¿Nadie en tu familia hace surf? Pero pensé que habías dicho que vivías en California antes de esto.,¿Nadie en tu familia hace surf? Pero tal vez vivieron en California en un momento de su vida y eso fue lo que te hizo pensar que lo hacían.


In [4]:
# Valores vacíos por columna
proto.isna().sum()

label                    0
fallacy_example          0
non-fallacy_example    765
dtype: int64

In [5]:
proto["label"].value_counts()

label
faulty generalization     399
ad hominem                258
ad populum                191
false causality           190
circular reasoning        158
deductive fallacy         141
appeal to emotion         140
fallacy of relevance      138
intentional               122
false dilemma             120
fallacy of credibility    118
fallacy of extension      108
Name: count, dtype: int64

In [6]:
# Longitud de las falacias en palabras
contar_palabras(proto["fallacy_example"]).describe()

count    2083.000000
mean       24.914066
std        17.068766
min         4.000000
25%        14.000000
50%        21.000000
75%        31.000000
max       193.000000
Name: fallacy_example, dtype: float64

In [7]:
# Textos repetidos
duplicadas = proto[proto["fallacy_example"].duplicated(keep=False)]
print("Filas con el texto repetido:", len(duplicadas))

# ¿Algún texto repetido tiene etiquetas distintas?
etiquetas_por_texto = duplicadas.groupby("fallacy_example")["label"].nunique()
print("Textos repetidos con etiquetas distintas:", (etiquetas_por_texto > 1).sum())

duplicadas.sort_values("fallacy_example").head(6)

Filas con el texto repetido: 48
Textos repetidos con etiquetas distintas: 1


,label,fallacy_example,non-fallacy_example
463,false causality,"""Ha llovido y hemos perdido el partido. Cada vez que llueve, vamos a perder"".","A veces llueve y perdemos el partido, pero no siempre es así."
484,false causality,"""Ha llovido y hemos perdido el partido. Cada vez que llueve, vamos a perder"".",No necesariamente vamos a perder cada vez que llueve.
1182,ad hominem,"Boris no está cualificado para hacer sugerencias sobre nuestro sistema penal. Como ex convicto, siempre se pondría del lado de los delincuentes.",NaN
1109,ad hominem,"Boris no está cualificado para hacer sugerencias sobre nuestro sistema penal. Como ex convicto, siempre se pondría del lado de los delincuentes.",NaN
414,false causality,"Cada vez que ese gallo canta, sale el sol. ¡Ese gallo debe ser muy poderoso e importante!","Cada vez que el gallo canta, es una señal de que se está acercando el amanecer. ¡Ese gallo debe tener un buen oído para saber cuándo llega el sol!"
581,false causality,"Cada vez que ese gallo canta, sale el sol. ¡Ese gallo debe ser muy poderoso e importante!",NaN


In [8]:
# Algunas falacias junto a su versión sin falacia
proto.dropna(subset=["non-fallacy_example"]).sample(5, random_state=0)

,label,fallacy_example,non-fallacy_example
1284,deductive fallacy,"Reconociendo que Amanda había cometido una falacia al argumentar que debíamos comer alimentos saludables porque un nutricionista decía que eran populares, Alyse dijo que, por lo tanto, debíamos co...","Reconociendo que Amanda había cometido una falacia al argumentar que debíamos comer alimentos saludables porque un nutricionista decía que eran populares, Alyse dijo que, por lo tanto, debíamos co..."
807,ad populum,"""Voy a comprar un iPhone porque todo el mundo tiene un iPhone y son geniales"".",Voy a comprar un iPhone porque me gusta el diseño y las características del teléfono.
1769,fallacy of relevance,"Señoras y señores del jurado, miren la ropa ensangrentada, el arma homicida. Imaginen los gritos de impotencia de la víctima. Un crimen así no merece otro veredicto que el de culpable, culpable.","El veredicto de un juicio debe basarse en pruebas y argumentos sólidos, en lugar de ser influenciado por emociones o apelaciones a la lógica."
486,false causality,"Lois y Jan hicieron un ""baile de la nieve"" una tarde, y esa noche nevó. Afirmaron haber traído la nieve.","Lois y Jan realizaron un ""baile de la nieve"" una tarde, pero no pueden afirmar haber traído la nieve."
1925,fallacy of credibility,"""Vamos a protestar y a no meternos en problemas porque el señor Iglesias ha dicho que está bien"".",Vamos a protestar pacíficamente y a no meternos en problemas porque creemos que es importante expresar nuestras opiniones.


## 3. FallacyES — falacias espontáneas

Cada fila es un fragmento de un comentario de meneame.net con su tipo (`label`, que vale `non-fallacy` si no hay falacia) y los datos de la noticia en la que se comentó. Los autores buscaron comentarios en los que un usuario acusaba a otro de una falacia y anotaron a mano el trozo falaz. Los textos sin falacia son fragmentos de otros comentarios de la misma noticia.

In [9]:
espont = pd.read_csv(CARPETA_FALLACYES / "FallacyES-spontaneous.csv")
print(espont.shape)
espont.head(3)

(1840, 6)


,label,text,source_link,news_title,news_summary,news_link
0,ad hominem,Para mi lo que diga este tipejo tiene cero validez.,https://old.meneame.net/story/como-milmillonarios-provocaron-caida-imperio-romano,Cómo los milmillonarios provocaron la caída del Imperio romano,"La causa del hundimiento del Imperio romano de Occidente fue la avaricia de las élites, las luchas intestinas que entablaron entre sí para alcanzar el poder y su creciente desapego del Estado, afi...",https://www.larazon.es/cultura/historia/20221009/w57unlibcneo7bjqbf4tjwn6vq.html?outputType=amp
1,ad hominem,pero era un HDGP o no?,https://old.meneame.net/story/gaddafi-hablando-sobre-invasion-eeuu-irak-asesinato-saddam,Gaddafi hablando sobre la invasión de EEUU a Irak y el asesinato de Saddam Hussein,"3 años antes de su muerte, advirtió a la Liga Árabe: ""Un líder árabe fue ejecutado en la horca. Cualquiera de ustedes podría ser el próximo. ""Somos amigos de Estados Unidos, pero uno de estos días...",https://twitter.com/RoiLopezRivas/status/1583845910043238400?s=20&t=6Ct7F_u-vMpPfZDoNu-o2Q
2,ad hominem,"El hombre, que acumula una veintena de antecedentes, asegura que los agentes fueron los que fueron a buscarlo y que se mearon en la puerta de su casa",https://old.meneame.net/story/mossos-fuera-sevicio-sabado-noche-c-roig-raval-catalan-tuit,MOSSOS FUERA DE SEVICIO el sábado por la noche en la C/Roig del Raval [En catalán el tuit. El vídeo en castellano.],"MOSSOS FUERA DE SEVICIO el sábado por la noche en la C/Roig del Raval. Según los vecinos: Al menos borrachos, robaron 2 patines, amenazaron a los vecinos, rompieron una puerta a patadas, mearon en...",https://twitter.com/SantiAranya/status/1579434149739663361


In [10]:
espont.isna().sum()

label           0
text            0
source_link     0
news_title      0
news_summary    0
news_link       0
dtype: int64

In [11]:
espont["label"].value_counts()

label
non-fallacy               917
ad hominem                208
fallacy of credibility    185
false dilemma             182
fallacy of extension      153
ad populum                 67
faulty generalization      48
deductive fallacy          42
appeal to emotion          38
Name: count, dtype: int64

In [12]:
print("Textos repetidos:", espont["text"].duplicated().sum())

# Longitud en palabras, con falacia (True) y sin ella (False)
espont["es_falacia"] = espont["label"] != "non-fallacy"
contar_palabras(espont["text"]).groupby(espont["es_falacia"]).describe()

Textos repetidos: 0


,count,mean,std,min,25%,50%,75%,max
es_falacia,,,,,,,,
False,917.0,25.960742,11.272749,6.0,18.0,24.0,32.0,73.0
True,923.0,26.764897,15.934450,3.0,16.0,23.0,33.0,129.0


In [13]:
# Las falacias y los textos sin falacia salen de las mismas noticias:
# importa para repartir train/test por noticia.
print("Noticias distintas:", espont["news_link"].nunique())
tipos_por_noticia = espont.groupby("news_link")["es_falacia"].nunique()
print("Noticias con textos con y sin falacia:", (tipos_por_noticia == 2).sum())

Noticias distintas: 826
Noticias con textos con y sin falacia: 462


In [14]:
# Rasgos de texto escrito informal que afectan a la limpieza y a la división en oraciones
textos = espont["text"]
print("Con saltos de línea:", textos.str.contains("\n").sum())
print("Sin signo final (. ! ? …):", (~textos.str.strip().str.contains(r"[.!?…]$")).sum())
print("Empiezan en minúscula:", textos.str.match(r"[a-záéíóúñ]").sum())

Con saltos de línea: 43
Sin signo final (. ! ? …): 230
Empiezan en minúscula: 112


In [15]:
espont[["label", "text"]].sample(8, random_state=3)

,label,text
1179,non-fallacy,La legislación actual impide el uso de hormonas para engordar ganado.
180,ad hominem,Tu aplaude que no das para mas
806,deductive fallacy,"un pais sin creencias y ritos, es un pais sin futuro, porque esa energia nos hace falta para seguir teniendo esperanza."
317,fallacy of credibility,Soy fisico y trabajo con modelos y defininiciones a diario!! Creo que estas predicando en el desierto la verdad! xD. Pero siempre estamos hablando con nosotros mismos!!!
1643,non-fallacy,"Supongo que depende de la comercializadora: El jueves pasado me pasé a la PVPC de Iberdrola (curenergia), y me llevó solamente alrededor de 10 minutos via web; y al dia siguiente (el viernes) ya s..."
818,faulty generalization,"Eso es una foto de un agujero en el uniforme, no es que no crea que no ha ocurrido, más aún viendo cómo les reciben a hostias y cómo hacen devoluciones en caliente, pero es que esa foto no es prue..."
815,faulty generalization,"mi hermana es castellana, lleva mas de 10 años viviendo en guipuzcoa y nunca ha tenido que callarse ni agachar la cabeza ante nadie. De hecho vive tan bien que no tiene ninguna intención de volver."
1015,non-fallacy,"Clara Serra es de las pocas personas en el entorno de Podemos sin una mentalidad dogmática, dispuesta a debatir con personas que no piensan como ella."


## 4. FallacyES-Political

Cada fila es un fragmento falaz de un debate electoral, con su tipo (`category`, a veces varios separados por coma), la fecha del debate (`document`), el orador, su partido y el texto de alrededor (`prev_context`, `foll_context`). **No hay ejemplos sin falacia.**

In [16]:
political = pd.read_csv(CARPETA_POLITICAL / "fallacyES-political.csv")
print(political.shape)
political.head(3)

(1954, 8)


,document,category,text,prev_context,foll_context,speaker_id,speaker_name,speaker_party
0,1993-05-24,False Cause,"Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista Y esa política, que ha sido una política arrogante, es la que ha gene...","Pueden ser también los jóvenes hoy desorientados, desanimados porque no encuentran un empleo, porque el precio de la vivienda es muy alto o simplemente porque no tienen la motivación para abrirse ...","\nPero si esas son las causas, yo quiero apostar en este momento por las soluciones.",SPEAKER_01,José María Aznar,PP
1,1993-05-24,Appeal to Emotion,"Porque efectivamente hoy los españoles tenemos ante nosotros la oportunidad de recuperar esperanza, de recuperar ilusión, de recuperar capacidad de trabajo. Y es por lo que creo que ha llegado la...","Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista\nY esa política, que ha sido una política arrogante, es la que ha gene...","\nUn gobierno eficaz, compuesto por personas competentes.",SPEAKER_01,José María Aznar,PP
2,1993-05-24,Ad Hominem,"Me permitirá decirle que con el historial de promesas incumplidas que usted tiene encima, de los españoles para volver a tener la sensibilidad necesaria para acercarse a los problemas de los ciu...","Yo quiero recordarle, en definitiva,\nque usted desde el comienzo tuvo un magnífico caudal de entusiasmo ciudadano.\nJamás un gobernante ha tenido, democrático, ha tenido el apoyo que usted ha ten...","\nPor eso hace falta el cambio ahora mismo.\n[CHANGE TO SPEAKER_02]\nEl problema sigue siendo creer o no creer en el país, sin duda alguna.",SPEAKER_01,José María Aznar,PP


In [17]:
political.isna().sum()

document          0
category          0
text              0
prev_context      1
foll_context      5
speaker_id       10
speaker_name     10
speaker_party    10
dtype: int64

In [18]:
# Filas con más de una etiqueta
es_multietiqueta = political["category"].str.contains(",")
print("Filas con más de una etiqueta:", es_multietiqueta.sum())
political.loc[es_multietiqueta, "category"].value_counts()

Filas con más de una etiqueta: 11


category
Appeal to Authority, False Cause       2
Appeal to Fear, Strawman               1
Appeal to Fear, Poisoning the Well     1
Complex Question, False Cause          1
Appeal to Emotion, False Cause         1
Appeal to Emotion, Complex Question    1
False Analogy, False Cause             1
Appeal to Fear, False Dilemma          1
Ad Hominem, False Dilemma              1
Appeal to Emotion, False Dilemma       1
Name: count, dtype: int64

In [19]:
# Recuento por etiqueta (una fila con dos etiquetas cuenta en las dos)
political["category"].str.split(", ").explode().value_counts()

category
Strawman                351
False Cause             304
Ad Hominem              285
False Dilemma           181
Appeal to Emotion       155
Poisoning the Well      152
Appeal to Fear          115
Appeal to Authority      87
Complex Question         74
False Analogy            67
Hasty Generalization     59
Ad Populum               50
Red Herring              28
Slippery Slope           23
Flag Waving              19
Slogan                   15
Name: count, dtype: int64

In [20]:
# Falacias por debate. La fecha (document) identifica cada debate.
debates = pd.read_csv(CARPETA_POLITICAL / "debates_info.csv")
falacias_por_debate = political["document"].value_counts().rename("falacias")
debates.set_index("Date").join(falacias_por_debate).sort_index()

,Participants,Organizer,falacias
Date,,,
1993-05-24,"José María Aznar (PP), Felipe González (PSOE)",Antena 3,88
2008-02-25,"Mariano Rajoy (PP), José Luis R. Zapatero (PSOE)",AcademiaTV,120
2008-03-03,"Mariano Rajoy (PP), José Luis R. Zapatero (PSOE)",AcademiaTV,110
2011-11-07,"Mariano Rajoy (PP), Alfredo Pérez Rubalcaba (PSOE)",AcademiaTV,83
2015-11-23,"Pablo Iglesias (Podemos), Albert Rivera (Cs)",Univ.Carlos III,51
2015-11-30,"Pedro Sánchez (PSOE), Pablo Iglesias (Podemos), Albert Rivera (Cs)",El País,69
2015-12-14,"Pedro Sánchez (PSOE), Mariano Rajoy (PP)",Atresmedia-AcademiaTV,50
2016-06-13,"Mariano Rajoy (PP), Pedro Sánchez (PSOE), Pablo Iglesias (Podemos), Albert Rivera (Cs)",AcademiaTV,89
2019-04-16,"Cayetana Álvarez de Toledo (PP), María Jesús Montero (PSOE), Irene Montero (UP), Inés Arrimadas (Cs), Gabriel Rufián (ERC), Aitor Esteban (PNV)",RTVE,86


In [21]:
political["speaker_party"].value_counts()

speaker_party
PP         591
PSOE       400
Cs         328
VOX        241
Podemos    119
ERC         61
UP          55
UNKNOWN     48
Sumar       43
Junts       32
PNV         18
EHBILDU      8
Name: count, dtype: int64

In [22]:
political["speaker_name"].value_counts()

speaker_name
Mariano Rajoy                    224
Albert Rivera                    212
Pedro Sánchez                    185
Pablo Iglesias                   119
Santiago Abascal                 118
Pablo Casado                     114
José Luis R. Zapatero             84
Inés Arrimadas                    78
Iván Espinosa de los Monteros     72
Gabriel Rufián                    61
Alberto Núñez Feijóo              60
José María Aznar                  57
Cayetana Álvarez de Toledo        55
MODERADOR                         44
Irene Montero                     41
Cuca Gamarra                      37
Alfredo Pérez Rubalcaba           33
Laura Borràs                      32
Rocío Monasterio                  32
Felipe González                   31
Teodoro García Egea               29
Yolanda Díaz                      23
Toni Cantó                        21
María Jesús Montero               20
Aina Vidal                        20
Javier Ortega Smith               19
Aitor Esteban            

In [23]:
# MODERADOR y DECLARACIONES no son candidatos: ¿qué falacias se les atribuyen?
no_candidatos = political[political["speaker_name"].isin(["MODERADOR", "DECLARACIONES"])]
print("Falacias de MODERADOR o DECLARACIONES:", len(no_candidatos))
no_candidatos[["document", "speaker_name", "category", "text"]].head()

Falacias de MODERADOR o DECLARACIONES: 48


,document,speaker_name,category,text
180,2015-11-30,MODERADOR,Appeal to Fear,"Pero antes de comenzar con el bloque de economía, noticia de estas últimas semanas, el terrorismo islamista, algo que nos preocupa mucho y que es verdaderamente grave. Hace unas semanas ha sido e..."
189,2015-11-30,MODERADOR,Appeal to Emotion,"Hoy muchos jóvenes que nos ven saben que aunque trabajen bien, aunque se esfuercen, se les acaba el contrato temporal y les echan y contratan a otro, como si fueran Kleenex. Y yo no quiero jóvene..."
221,2015-12-14,MODERADOR,False Cause,"Absolutamente, pero para eso necesitamos un cambio de gobierno. Porque si algo ha demostrado precisamente este gobierno es que no sé si podremos lograr la recuperación económica con la derecha, ..."
235,2015-12-14,MODERADOR,Red Herring,"La política exterior ya es política nacional. La política europea es política nacional. Lo que pasa, por ejemplo, o lo que ha ocurrido en Kabul, y desde luego todo mi reconocimiento a los policí..."
236,2015-12-14,MODERADOR,Red Herring,Vamos a seguir en este ámbito en el que estamos hablando. ¿Tienen algunos planes para relanzar la economía española también mirando al exterior? Estamos en un ámbito en el que nos conformamos co...


In [24]:
contar_palabras(political["text"]).describe()

count    1954.000000
mean       41.534800
std        38.622228
min         3.000000
25%        20.000000
50%        30.000000
75%        50.000000
max       412.000000
Name: text, dtype: float64

In [25]:
# Restos de la transcripción automática (WhisperX):
# marcas de cambio de orador, p. ej. "[CHANGE TO SPEAKER_05]", y dobles espacios
marca_orador = r"\[CHANGE TO [^\]]*\]"
for columna in ["text", "prev_context", "foll_context"]:
    print(f"{columna}: {political[columna].str.contains(marca_orador).sum()} con marcas de orador")
print("text con dobles espacios:", political["text"].str.contains("  ").sum())

text: 9 con marcas de orador
prev_context: 898 con marcas de orador
foll_context: 885 con marcas de orador
text con dobles espacios: 1054


In [26]:
political[["document", "speaker_name", "category", "text"]].sample(5, random_state=1)

,document,speaker_name,category,text
1812,2023-07-13,Iván Espinosa de los Monteros,Hasty Generalization,"A los españoles les preocupa la educación de sus hijos y les preocupa que cada vez sea peor, que cada vez sea menos exigente, que se haya quebrado lo que era, sin duda, el mejor ascensor social, q..."
1576,2019-11-02,Laura Borràs,Ad Hominem,"Felipe VI, el rey del ""a por ellos"", iba a ser el último rey en abusar de los catalanes, el último borbón en abusar de los catalanes"
1926,2023-07-19,Pedro Sánchez,Poisoning the Well,"no es que le dé vergüenza los votos del señor Abascal, que los asume. Tampoco sea vergüenza de las políticas del señor Abascal, que las asume también. Tampoco sea vergüenza de gobernar con Vox, ..."
1378,2019-04-22,Pablo Casado,False Cause,Y si yo gané las primarias de mi partido fue porque tuve un comportamiento muy firme contra cualquier caso de corrupción.
1446,2019-04-23,Albert Rivera,Flag Waving,que sueñen en la España que queremos. Los españoles sois mi gran familia también. Todos vosotros sois parte de una gran familia que es España.


## 5. Resumen de la exploración

| | Prototípicas | Espontáneas | Political |
|---|---|---|---|
| Filas | 2.083 | 1.840 | 1.954 |
| Clases | 12 | 8 + `non-fallacy` | 16 |
| Sin falacia | 1.318 parejas escritas a mano | 917 textos reales | ninguno |
| Palabras por texto (media) | 25 | 26 | 42 |
| Multietiqueta | no | no | 11 filas |

Lo que habrá que tratar en los siguientes pasos:

- **Prototípicas**: 48 filas tienen un texto que aparece más de una vez (y uno de esos textos tiene dos etiquetas distintas): hay que quitar los duplicados antes de hacer los splits. 765 falacias no tienen pareja sin falacia.
- **Espontáneas**: es texto escrito informal: 43 textos con saltos de línea, 230 sin signo final y 112 que empiezan en minúscula. Afecta a la división en oraciones. Las falacias y los textos sin falacia salen de las mismas noticias (462 de 826), lo que confirma que los splits deben hacerse por noticia.
- **Political**:
  - Restos de la transcripción automática: dobles espacios en 1.054 textos (donde la transcripción no puso punto) y marcas de cambio de orador (`[CHANGE TO SPEAKER_05]`) en 9 textos y en casi la mitad de los contextos.
  - El orador no siempre es fiable: 10 falacias no tienen orador y 48 se atribuyen a `MODERADOR` o `DECLARACIONES`, aunque varias son claramente de un candidato (p. ej. *"necesitamos un cambio de gobierno"*). Parecen errores al separar quién habla en la transcripción. Para el análisis por político habrá que descartarlas o revisarlas.
  - Un nombre viene mal escrito en el original: `Adrana Lastra` (Adriana Lastra).

## 6. Etiquetas: de las originales a las claves comunes

Cada corpus tiene su propia lista de falacias (12 tipos en las prototípicas, 8 en las espontáneas y 16 en Political). Para poder usarlos juntos, cada etiqueta original se traduce a una **clave común**: 10 clases más `otra_falacia`. Se guardan las dos cosas:

- `etiquetas_original`: la etiqueta tal como viene en el corpus.
- `etiquetas`: la clave común. Es la que usarán los modelos.

Así, si el grupo decide cambiar la tabla, basta con volver a aplicarla. Las dos son listas: en Political algunos fragmentos tienen varias etiquetas, y un texto sin falacia tiene la lista vacía `[]`.

Decisiones de la tabla:
- **Correspondencias directas**: las que dan los propios autores de los corpus (p. ej., *fallacy of extension* de FallacyES es el *Strawman* de Political).
- **Fusiones propias** (tipos de Political que no existen en FallacyES y se meten en una clase parecida): *Poisoning the Well* → `ad_hominem` (la guía de Political la define como "una forma intensificada de ad hominem"); *Appeal to Fear* y *Flag Waving* → `apelacion_emocion` (son apelaciones a emociones concretas: miedo y sentimiento de grupo); *Appeal to Authority* → `falacia_credibilidad` (en FallacyES la credibilidad incluye la apelación a la autoridad); *False Analogy* → `deduccion_invalida` (en FallacyES la deducción errónea incluye la falsa analogía).
- **`otra_falacia`**: las falacias que solo aparecen en un corpus (*circular reasoning* e *intentional* en las prototípicas; *Complex Question*, *Slippery Slope* y *Slogan* en Political). Siguen contando como "hay falacia" para la detección, pero no hay ejemplos suficientes en los demás corpus para tratarlas como clases propias.

In [27]:
# Etiqueta original (en minúsculas) → clave común.
# FallacyES escribe las etiquetas en minúscula ("ad hominem") y Political con mayúsculas
# ("Ad Hominem"), por eso se comparan siempre en minúsculas.
CLAVE_COMUN = {
    "ad hominem": "ad_hominem",
    "poisoning the well": "ad_hominem",                  # fusión (Political)
    "fallacy of extension": "hombre_de_paja",            # FallacyES
    "strawman": "hombre_de_paja",                        # Political
    "faulty generalization": "generalizacion_apresurada",  # FallacyES
    "hasty generalization": "generalizacion_apresurada",   # Political
    "false causality": "falsa_causalidad",               # FallacyES
    "false cause": "falsa_causalidad",                   # Political
    "false dilemma": "falso_dilema",
    "appeal to emotion": "apelacion_emocion",
    "appeal to fear": "apelacion_emocion",               # fusión (Political)
    "flag waving": "apelacion_emocion",                  # fusión (Political)
    "fallacy of credibility": "falacia_credibilidad",    # FallacyES
    "appeal to authority": "falacia_credibilidad",       # fusión (Political)
    "ad populum": "ad_populum",
    "deductive fallacy": "deduccion_invalida",           # FallacyES
    "false analogy": "deduccion_invalida",               # fusión (Political)
    "fallacy of relevance": "distraccion",               # FallacyES
    "red herring": "distraccion",                        # Political
    "circular reasoning": "otra_falacia",
    "intentional": "otra_falacia",
    "complex question": "otra_falacia",
    "slippery slope": "otra_falacia",
    "slogan": "otra_falacia",
}


def separar_etiquetas(etiqueta_csv):
    """Pasa la etiqueta del CSV a lista. "non-fallacy" (sin falacia) da la lista vacía;
    en Political puede haber varias separadas por coma: "Appeal to Fear, Strawman"."""
    if etiqueta_csv == "non-fallacy":
        return []
    return [etiqueta.strip() for etiqueta in etiqueta_csv.split(",")]


def a_claves_comunes(etiquetas_original):
    """Traduce cada etiqueta original a su clave común, sin repetir claves."""
    claves = []
    for etiqueta in etiquetas_original:
        clave = CLAVE_COMUN[etiqueta.lower()]
        if clave not in claves:
            claves.append(clave)
    return claves

Antes de aplicar la tabla, se comprueba que todas las etiquetas que aparecen en los tres corpus tienen una clave común:

In [28]:
todas_las_etiquetas = (
    pd.concat([proto["label"], espont["label"], political["category"]])
    .map(separar_etiquetas)
    .explode()   # una fila por etiqueta
    .dropna()    # los textos sin falacia dan una lista vacía
)
sin_clave = sorted(set(todas_las_etiquetas[~todas_las_etiquetas.str.lower().isin(CLAVE_COMUN)]))
print("Etiquetas distintas en los tres corpus:", todas_las_etiquetas.nunique())
print("Etiquetas sin clave común:", sin_clave if sin_clave else "ninguna")

Etiquetas distintas en los tres corpus: 28
Etiquetas sin clave común: ninguna


Se aplica la tabla a los tres corpus. En las prototípicas, la etiqueta es la de la falacia (`fallacy_example`); su pareja sin falacia tendrá `[]` cuando se separen en dos ejemplos (más adelante).

In [29]:
proto["etiquetas_original"] = proto["label"].map(separar_etiquetas)
espont["etiquetas_original"] = espont["label"].map(separar_etiquetas)
political["etiquetas_original"] = political["category"].map(separar_etiquetas)

for corpus in [proto, espont, political]:
    corpus["etiquetas"] = corpus["etiquetas_original"].map(a_claves_comunes)

political.loc[es_multietiqueta, ["category", "etiquetas_original", "etiquetas"]]

,category,etiquetas_original,etiquetas
329,"Appeal to Fear, Strawman","[Appeal to Fear, Strawman]","[apelacion_emocion, hombre_de_paja]"
469,"Appeal to Fear, Poisoning the Well","[Appeal to Fear, Poisoning the Well]","[apelacion_emocion, ad_hominem]"
839,"Complex Question, False Cause","[Complex Question, False Cause]","[otra_falacia, falsa_causalidad]"
869,"Appeal to Emotion, False Cause","[Appeal to Emotion, False Cause]","[apelacion_emocion, falsa_causalidad]"
915,"Appeal to Emotion, Complex Question","[Appeal to Emotion, Complex Question]","[apelacion_emocion, otra_falacia]"
1020,"False Analogy, False Cause","[False Analogy, False Cause]","[deduccion_invalida, falsa_causalidad]"
1062,"Appeal to Authority, False Cause","[Appeal to Authority, False Cause]","[falacia_credibilidad, falsa_causalidad]"
1090,"Appeal to Authority, False Cause","[Appeal to Authority, False Cause]","[falacia_credibilidad, falsa_causalidad]"
1504,"Appeal to Fear, False Dilemma","[Appeal to Fear, False Dilemma]","[apelacion_emocion, falso_dilema]"
1605,"Ad Hominem, False Dilemma","[Ad Hominem, False Dilemma]","[ad_hominem, falso_dilema]"


In [30]:
# Ejemplos por clave común en cada corpus (una fila con dos etiquetas cuenta en las dos)
recuento = pd.DataFrame({
    "prototípicas": proto["etiquetas"].explode().value_counts(),
    "espontáneas": espont["etiquetas"].explode().value_counts(),
    "political": political["etiquetas"].explode().value_counts(),
}).fillna(0).astype(int)
recuento["total"] = recuento.sum(axis=1)
recuento.sort_values("total", ascending=False)

,prototípicas,espontáneas,political,total
etiquetas,,,,
ad_hominem,258,208,437,903
hombre_de_paja,108,153,351,612
generalizacion_apresurada,399,48,59,506
falsa_causalidad,190,0,304,494
falso_dilema,120,182,181,483
apelacion_emocion,140,38,289,467
otra_falacia,280,0,112,392
falacia_credibilidad,118,185,87,390
ad_populum,191,67,50,308


In [31]:
# Comprobaciones: los textos sin falacia (y solo ellos) tienen la lista vacía,
# y ninguna fila multietiqueta de Political se ha quedado con una sola clave.
sin_falacia = espont["etiquetas"].str.len() == 0
print("Espontáneas con lista vacía:", sin_falacia.sum(), "| con label non-fallacy:", (espont["label"] == "non-fallacy").sum())
print("Prototípicas con lista vacía:", (proto["etiquetas"].str.len() == 0).sum())
print("Political con lista vacía:", (political["etiquetas"].str.len() == 0).sum())
fusionadas = political["etiquetas"].str.len() < political["etiquetas_original"].str.len()
print("Filas de Political cuyas etiquetas se han juntado en una sola clave:", fusionadas.sum())

Espontáneas con lista vacía: 917 | con label non-fallacy: 917
Prototípicas con lista vacía: 0
Political con lista vacía: 0
Filas de Political cuyas etiquetas se han juntado en una sola clave: 0


Resultado: todas las etiquetas tienen clave común, los textos sin falacia quedan con `[]` y las 11 filas multietiqueta de Political conservan dos claves (ninguna combinaba dos tipos que vayan a la misma clase). La clase más pequeña es `distraccion` (166 ejemplos) y la más grande `ad_hominem` (903): hay desequilibrio entre clases, a tener en cuenta al entrenar y al evaluar (por ejemplo, con F1 macro).

## 7. Duplicados y parejas sin falacia que no sirven

Dos problemas de calidad que hay que resolver antes de hacer los splits: si el mismo texto queda en train y en test, el modelo "se lo sabe" y las métricas salen infladas.

Para comparar textos se usa una versión **normalizada** (minúsculas, sin signos de puntuación y con un solo espacio entre palabras). Así, *"O me amas, o me odias"*, *"O me amas o me odias."* y *"\"O me amas, o me odias\""* cuentan como el mismo texto. La versión normalizada solo sirve para comparar: el texto que se guarda no cambia.

El orden importa: primero se quitan las parejas sin falacia que no sirven (7.1) y después los duplicados (7.2). Al quitar duplicados se elige la copia que tiene pareja, y así nunca se elige una copia cuya pareja se iba a quitar.

In [32]:
import re
from difflib import SequenceMatcher


def normalizar_para_comparar(texto):
    """Minúsculas, sin signos de puntuación y con un solo espacio entre palabras.
    Solo se usa para comparar textos: el texto que se guarda no cambia."""
    texto = re.sub(r"[^\w\s]", "", texto.lower())
    return " ".join(texto.split())


def parecido(texto_a, texto_b):
    """Parecido entre dos textos normalizados, de 0 (nada) a 1 (iguales)."""
    return SequenceMatcher(None, normalizar_para_comparar(texto_a), normalizar_para_comparar(texto_b)).ratio()

### 7.1 Parejas sin falacia que no quitan la falacia (prototípicas)

Las parejas sin falacia se escribieron modificando la falacia, a veces con un cambio mínimo y válido (*"Todas las clases de matemáticas deben ser difíciles"* → *"**No** todas…"*). Pero en algunas el cambio no quita la falacia, o no hay cambio. Esas parejas no pueden quedarse: el mismo argumento estaría etiquetado como falacia y como no falacia.

Se buscan las parejas que se parecen más de un 95 % a su falacia:

In [33]:
con_pareja = proto.dropna(subset=["non-fallacy_example"])
parecido_con_su_falacia = con_pareja.apply(
    lambda fila: parecido(fila["fallacy_example"], fila["non-fallacy_example"]), axis=1
)
muy_parecidas = parecido_con_su_falacia[parecido_con_su_falacia >= 0.95]
print("Parejas con un parecido de al menos el 95 %:", len(muy_parecidas))
print("De ellas, idénticas a su falacia:", (muy_parecidas == 1).sum())

Parejas con un parecido de al menos el 95 %: 35
De ellas, idénticas a su falacia: 5


Las idénticas se quitan siempre. Las demás se revisaron a mano una a una: muchas son parejas mínimas válidas (*"Todos…"* → *"No todos…"*, *"bichos"* → *"simios"*), pero en 12 el cambio no quita la falacia. Esas 12 se apuntan aquí con el motivo (el número es la fila del CSV original).

Limitación: solo se revisaron las parejas con más de un 95 % de parecido; puede quedar alguna pareja mal hecha menos parecida.

In [34]:
# Revisadas a mano: el cambio de la pareja no quita la falacia.
PAREJAS_QUE_NO_QUITAN_LA_FALACIA = {
    35: "'Pero cuando' → 'Cuando': no cambia el argumento",
    2011: "solo quita 'Ejemplo #2:'",
    1517: "solo cambia la puntuación ('demás, pero no te vi' → 'demás. No te vi')",
    1521: "'comunistas' → 'socialistas': sigue siendo un falso dilema",
    1745: "solo quita 'aún peor': sigue desviando la pregunta",
    1890: "'superiores' → 'buenos': el argumento es el mismo",
    639: "'Todos los jóvenes' → 'Los jóvenes': sigue siendo circular",
    695: "'apoyo público' → 'el apoyo público suficiente': sigue siendo circular",
    1667: "'no puede creer' → 'es sorprendente': sigue tergiversando a Pamela",
    1302: "'son chicos' → 'no son chicos': el silogismo sigue siendo inválido",
    479: "'mueven' → 'mueven bruscamente': no cambia nada",
    1781: "'son' → 'pueden ser': matiz que no toca la desviación de tema (dudoso)",
}

pareja_identica = muy_parecidas[muy_parecidas == 1].index
quitar_pareja = proto.index.isin(pareja_identica) | proto.index.isin(PAREJAS_QUE_NO_QUITAN_LA_FALACIA)

proto.loc[quitar_pareja, "non-fallacy_example"] = None
print("Parejas quitadas:", quitar_pareja.sum(), "| parejas que quedan:", proto["non-fallacy_example"].notna().sum())

Parejas quitadas: 17 | parejas que quedan: 1301


### 7.2 Duplicados

**Prototípicas.** De cada grupo de textos iguales se queda una sola fila: la primera que tenga pareja sin falacia o, si ninguna la tiene, la primera. Las parejas de las copias descartadas se pierden (son pocas y son frases escritas a mano). Un texto aparece con dos etiquetas: *"Dirigir el gobierno es como dirigir un negocio…"* (*faulty generalization* y *deductive fallacy*); se queda como *deductive fallacy*, porque es una analogía y en FallacyES la falsa analogía está dentro de *deductive fallacy*. Da la casualidad de que es la copia que tiene pareja, así que la regla general ya la elige; se comprueba abajo.

In [35]:
proto["texto_comparable"] = proto["fallacy_example"].map(normalizar_para_comparar)
proto["tiene_pareja"] = proto["non-fallacy_example"].notna()

copias = proto[proto["texto_comparable"].duplicated(keep=False)]
print("Textos repetidos:", copias["texto_comparable"].nunique(), "| filas afectadas:", len(copias))

# Se ordena para que, dentro de cada grupo, quede primero la fila con pareja (y, entre iguales,
# la de número más bajo: la ordenación "stable" respeta el orden original).
# drop_duplicates se queda con la primera fila de cada grupo.
filas_antes = len(proto)
proto = (
    proto.sort_values("tiene_pareja", ascending=False, kind="stable")
    .drop_duplicates("texto_comparable")
    .sort_index()
)
print(f"Filas: {filas_antes} → {len(proto)} (quitadas {filas_antes - len(proto)})")

Textos repetidos: 37 | filas afectadas: 80
Filas: 2083 → 2040 (quitadas 43)


In [36]:
# Comprobaciones
print("Textos repetidos que quedan:", proto["texto_comparable"].duplicated().sum())
conflicto = proto[proto["texto_comparable"].str.startswith("dirigir el gobierno es como dirigir un negocio")]
print("Etiqueta que se queda en el texto con dos etiquetas:", conflicto["label"].tolist())
print("Parejas sin falacia que quedan:", proto["tiene_pareja"].sum())

Textos repetidos que quedan: 0
Etiqueta que se queda en el texto con dos etiquetas: ['deductive fallacy']
Parejas sin falacia que quedan: 1285


**Espontáneas.** No tienen textos repetidos, ni siquiera normalizados:

In [37]:
print("Espontáneas repetidas:", espont["text"].map(normalizar_para_comparar).duplicated().sum())

Espontáneas repetidas: 0


**Political.** Hay 3 textos repetidos. Antes de quitarlos se comprueba que son de verdad el mismo fragmento anotado dos veces (mismo debate, orador y etiqueta), y no una frase que un político repite en debates distintos, que sería un ejemplo legítimo:

In [38]:
political["texto_comparable"] = political["text"].map(normalizar_para_comparar)
repetidos = political[political["texto_comparable"].duplicated(keep=False)]
repetidos[["document", "speaker_name", "category", "text"]]

,document,speaker_name,category,text
793,2023-07-19,Santiago Abascal,Appeal to Fear,"lo que hace, entre otras cosas, es corromper a los niños más pequeños en los colegios. Porque entran los activistas a contarles cosas que los niños no deberían conocer y a decirles"
795,2023-07-19,Santiago Abascal,Appeal to Fear,"lo que hace, entre otras cosas, es corromper a los niños más pequeños en los colegios. Porque entran los activistas a contarles cosas que los niños no deberían conocer y a decirles"
951,2008-03-03,Mariano Rajoy,Appeal to Fear,"Tampoco va bien la seguridad ciudadana y hay que afrontar la realidad, crece la delincuencia y hay muchas personas y muchas bandas organizadas que actúan de forma especialmente violenta."
952,2008-03-03,Mariano Rajoy,Appeal to Fear,"Tampoco va bien la seguridad ciudadana y hay que afrontar la realidad, crece la delincuencia y hay muchas personas y muchas bandas organizadas que actúan de forma especialmente violenta."
965,2008-03-03,Mariano Rajoy,False Dilemma,"Las personas que vienen de fuera tienen derechos, pero los españoles también tienen derechos, y a la hora de ir a una plaza escolar, a la hora de ver qué se puede hacer con las becas comedor, a la..."
966,2008-03-03,Mariano Rajoy,False Dilemma,"Las personas que vienen de fuera tienen derechos, pero los españoles también tienen derechos, y a la hora de ir a una plaza escolar, a la hora de ver qué se puede hacer con las becas comedor, a la..."


In [39]:
# Los tres son del mismo debate, con el mismo orador y la misma etiqueta: se queda la primera copia.
filas_antes = len(political)
political = political.drop_duplicates("texto_comparable")
print(f"Filas: {filas_antes} → {len(political)} (quitadas {filas_antes - len(political)})")

Filas: 1954 → 1951 (quitadas 3)


**Political: fragmentos anidados (se dejan como están).** Además de los duplicados, hay fragmentos cortos que están dentro de otro fragmento más largo del mismo debate, casi siempre con otra etiqueta. Por ejemplo, una intervención de Inés Arrimadas (16-04-2019) está anotada entera como *Poisoning the Well*, y dos trozos de ella están anotados aparte como *Appeal to Fear* (*"Sánchez es un peligro público y hay que sacarle a votos de Moncloa"*) y *False Dilemma*.

No son duplicados, sino falacias distintas anotadas sobre trozos que se solapan, así que no se quitan. No hay fuga entre train y test porque los splits de Political se hacen por debate entero y los dos trozos van siempre juntos. **Limitación**: el fragmento largo contiene falacias que no están en su etiqueta. No se corrige para no cambiar la anotación original.

In [40]:
# Pares (fragmento corto, fragmento largo que lo contiene) dentro del mismo debate
anidados = []
for fecha, debate in political.groupby("document"):
    for fila_corto, corto in debate["texto_comparable"].items():
        for fila_largo, largo in debate["texto_comparable"].items():
            if fila_corto != fila_largo and corto in largo:
                anidados.append((fila_corto, fila_largo))

print("Fragmentos dentro de otro más largo del mismo debate:", len(anidados))
con_otra_etiqueta = [(c, l) for c, l in anidados if political.loc[c, "category"] != political.loc[l, "category"]]
print("De ellos, con otra etiqueta:", len(con_otra_etiqueta))

Fragmentos dentro de otro más largo del mismo debate: 45
De ellos, con otra etiqueta: 37


### 7.3 Textos rotos

Ningún texto está vacío (se vio en la exploración). Al revisar los duplicados aparecieron 3 "falacias" de las prototípicas que en realidad son títulos de ejercicio del material original (empiezan por `SM` y un número) y no contienen ningún argumento. Se quitan con su pareja. La pareja de la fila 1665 era además una copia de la de la fila 1776, así que con esto también desaparece esa repetición.

No se ha hecho una búsqueda sistemática de más textos rotos: el ruido que pueda quedar es muy pequeño y está en el corpus que solo se usa para entrenar.

In [41]:
# Títulos de ejercicio, no falacias. Número = fila del CSV original.
TITULOS_DE_EJERCICIO = [1665, 1666, 1759]

print(proto.loc[TITULOS_DE_EJERCICIO, ["label", "fallacy_example"]])
proto = proto.drop(index=TITULOS_DE_EJERCICIO)
print("Filas de las prototípicas:", len(proto), "| con pareja sin falacia:", proto["non-fallacy_example"].notna().sum())

                     label  \
1665  fallacy of extension   
1666  fallacy of extension   
1759  fallacy of relevance   

                                                       fallacy_example  
1665                      SM 1: Debate sobre la Segunda Guerra Mundial  
1666              SM 6: El Norte invade el Sur durante la Guerra Civil  
1759  SM 5: Argumento sobre la locura como defensa contra la ejecución  
Filas de las prototípicas: 2037 | con pareja sin falacia: 1282


## 8. Prototípicas: un ejemplo por texto

En el CSV, cada fila tiene una falacia y, a veces, su pareja sin falacia. En el formato de salida cada texto es un ejemplo, así que cada fila da uno o dos ejemplos:

| Ejemplo | ID | Texto | `etiquetas_original` / `etiquetas` |
|---|---|---|---|
| La falacia | `fpr_0042_f` | `fallacy_example` | las de la fila (sección 6) |
| Su pareja sin falacia | `fpr_0042_n` | `non-fallacy_example` | `[]` / `[]` |

El número es la fila del CSV original, así que la falacia y su pareja comparten número. Se guarda también en `fila_original` porque hará falta para los splits: **la falacia y su pareja tienen que ir siempre al mismo split**, porque son casi el mismo texto.

In [42]:
def id_prototipica(fila, tipo):
    """ID de un ejemplo prototípico: nº de fila del CSV original + "f" (falacia) o "n" (sin falacia)."""
    return f"fpr_{fila:04d}_{tipo}"


falacias = pd.DataFrame({
    "id": [id_prototipica(fila, "f") for fila in proto.index],
    "fila_original": proto.index,
    "texto_original": proto["fallacy_example"].tolist(),
    "etiquetas_original": proto["etiquetas_original"].tolist(),
    "etiquetas": proto["etiquetas"].tolist(),
})

con_pareja = proto[proto["non-fallacy_example"].notna()]
sin_falacia = pd.DataFrame({
    "id": [id_prototipica(fila, "n") for fila in con_pareja.index],
    "fila_original": con_pareja.index,
    "texto_original": con_pareja["non-fallacy_example"].tolist(),
    "etiquetas_original": [[] for _ in con_pareja.index],
    "etiquetas": [[] for _ in con_pareja.index],
})

# Ordenado por ID: cada falacia queda justo encima de su pareja
proto_ejemplos = pd.concat([falacias, sin_falacia]).sort_values("id").reset_index(drop=True)
print(f"Falacias: {len(falacias)} | sin falacia: {len(sin_falacia)} | total: {len(proto_ejemplos)}")
proto_ejemplos.head(6)

Falacias: 2037 | sin falacia: 1282 | total: 3319


,id,fila_original,texto_original,etiquetas_original,etiquetas
0,fpr_0000_f,0,"""A Annie le debe gustar Starbucks porque a todas las chicas blancas les gusta Starbucks"".",[faulty generalization],[generalizacion_apresurada]
1,fpr_0000_n,0,A Annie le debe gustar Starbucks porque a ella le gustan las bebidas dulces y Starbucks tiene muchas opciones de bebidas dulces.,[],[]
2,fpr_0001_f,1,"Este año hace más calor en Las Vegas que el año pasado; por lo tanto, el calentamiento global se está acelerando rápidamente.",[faulty generalization],[generalizacion_apresurada]
3,fpr_0001_n,1,"El año pasado hacía más calor en Las Vegas que este año; por lo tanto, el calentamiento global se está ralentizando.",[],[]
4,fpr_0002_f,2,"""Los dos cursos que tomé en la UF no fueron muy interesantes. No creo que sea una buena universidad"".",[faulty generalization],[generalizacion_apresurada]
5,fpr_0002_n,2,Los dos cursos que tomé en la UF me parecieron aburridos. No tengo una opinión formada sobre si la UF es una buena universidad o no.,[],[]


In [43]:
# Comprobaciones
print("IDs repetidos:", proto_ejemplos["id"].duplicated().sum())
print("Parejas sin su falacia:", (~sin_falacia["fila_original"].isin(falacias["fila_original"])).sum())
print("Textos vacíos:", (proto_ejemplos["texto_original"].str.strip() == "").sum())
es_falacia = proto_ejemplos["etiquetas"].str.len() > 0
print("Ejemplos con falacia:", es_falacia.sum(), "| sin falacia:", (~es_falacia).sum())

IDs repetidos: 0
Parejas sin su falacia: 0
Textos vacíos: 0
Ejemplos con falacia: 2037 | sin falacia: 1282


## 9. Splits: train, dev y test

Ningún corpus trae los splits hechos, así que se crean aquí: más o menos **70 % train, 15 % dev y 15 % test**.

La regla es que dos textos muy parecidos no pueden quedar uno en train y otro en test (el modelo "se sabría" la respuesta y las métricas saldrían infladas). Por eso no se reparten textos sueltos, sino **grupos** que van enteros al mismo split:

| Corpus | Se reparte por | Por qué |
|---|---|---|
| Prototípicas | falacia + su pareja sin falacia | Son casi el mismo texto |
| Espontáneas | noticia | Los comentarios de una noticia hablan de lo mismo (y las falacias y los textos sin falacia salen de las mismas noticias) |
| Political | debate | Los fragmentos de un debate se repiten temas, frases y hasta se solapan (sección 7) |

En las prototípicas y las espontáneas el reparto es al azar con una semilla fija, para que salga siempre igual. En las prototípicas además se mantiene la proporción de cada tipo de falacia en los tres splits.

In [44]:
from sklearn.model_selection import train_test_split

SEMILLA = 42  # semilla fija: el reparto sale igual cada vez que se ejecuta


def repartir_70_15_15(grupos, columna_clase=None):
    """Recibe una tabla con una fila por grupo y devuelve el split de cada fila:
    train (70 %), dev (15 %) o test (15 %). Si se indica columna_clase, se mantiene
    la proporción de esa clase en los tres splits."""
    clases = grupos[columna_clase] if columna_clase else None
    train, resto = train_test_split(grupos, test_size=0.30, stratify=clases, random_state=SEMILLA)

    clases_resto = resto[columna_clase] if columna_clase else None
    dev, test = train_test_split(resto, test_size=0.50, stratify=clases_resto, random_state=SEMILLA)

    split = pd.Series("train", index=grupos.index)
    split[dev.index] = "dev"
    split[test.index] = "test"
    return split

**Prototípicas.** Se reparten las falacias (una por fila del CSV original), manteniendo la proporción de cada tipo, y cada pareja sin falacia hereda el split de su falacia:

In [45]:
falacias["clase"] = falacias["etiquetas"].str[0]  # en las prototípicas hay una etiqueta por falacia
falacias["split"] = repartir_70_15_15(falacias, columna_clase="clase")

split_de_cada_fila = dict(zip(falacias["fila_original"], falacias["split"]))
proto_ejemplos["split"] = proto_ejemplos["fila_original"].map(split_de_cada_fila)
proto_ejemplos["split"].value_counts()

split
train    2341
test      489
dev       489
Name: count, dtype: int64

**Espontáneas.** Se reparten las noticias al azar, y cada texto hereda el split de su noticia:

In [46]:
noticias = pd.DataFrame({"enlace": espont["news_link"].unique()})
noticias["split"] = repartir_70_15_15(noticias)

espont["split"] = espont["news_link"].map(dict(zip(noticias["enlace"], noticias["split"])))
espont["split"].value_counts()

split
train    1301
dev       273
test      266
Name: count, dtype: int64

**Political.** Se reparte por debate, y no al azar sino por fecha:
- **test**: los 3 debates de 2023, los más recientes.
- **dev**: los 2 debates más recientes antes de esos (4 y 7 de noviembre de 2019).
- **train**: el resto (1993–2019).

Así se entrena con debates anteriores y se evalúa con los más nuevos, que es lo que pasará al usar el detector con debates futuros. Ojo: algunos partidos (Sumar) solo aparecen en test.

In [47]:
DEBATES_TEST = ["2023-07-10", "2023-07-13", "2023-07-19"]
DEBATES_DEV = ["2019-11-04", "2019-11-07"]


def split_de_debate(fecha):
    if fecha in DEBATES_TEST:
        return "test"
    if fecha in DEBATES_DEV:
        return "dev"
    return "train"


political["split"] = political["document"].map(split_de_debate)
political["split"].value_counts()

split
train    1316
test      344
dev       291
Name: count, dtype: int64

### Comprobaciones

Tamaño de cada split (en %), que ningún grupo esté repartido entre dos splits y que ningún texto aparezca en dos splits:

In [48]:
ORDEN_SPLITS = ["train", "dev", "test"]
tamanos = pd.DataFrame({
    "prototípicas": proto_ejemplos["split"].value_counts(normalize=True),
    "espontáneas": espont["split"].value_counts(normalize=True),
    "political": political["split"].value_counts(normalize=True),
}).loc[ORDEN_SPLITS]
(tamanos * 100).round(1)

,prototípicas,espontáneas,political
split,,,
train,70.5,70.7,67.5
dev,14.7,14.8,14.9
test,14.7,14.5,17.6


In [49]:
print("Prototípicas: filas en más de un split:", (proto_ejemplos.groupby("fila_original")["split"].nunique() > 1).sum())
print("Espontáneas: noticias en más de un split:", (espont.groupby("news_link")["split"].nunique() > 1).sum())
print("Political: debates en más de un split:", (political.groupby("document")["split"].nunique() > 1).sum())

# Mismo texto (normalizado) en dos splits distintos
for nombre, textos, splits in [
    ("Prototípicas", proto_ejemplos["texto_original"], proto_ejemplos["split"]),
    ("Espontáneas", espont["text"], espont["split"]),
    ("Political", political["text"], political["split"]),
]:
    tabla = pd.DataFrame({"texto": textos.map(normalizar_para_comparar), "split": splits})
    print(f"{nombre}: textos en más de un split:", (tabla.groupby("texto")["split"].nunique() > 1).sum())

Prototípicas: filas en más de un split: 0
Espontáneas: noticias en más de un split: 0
Political: debates en más de un split: 0
Prototípicas: textos en más de un split: 0
Espontáneas: textos en más de un split: 0
Political: textos en más de un split: 0


Proporción de cada clase dentro de cada split (en %). Si el reparto es bueno, las tres columnas se parecen. `sin_falacia` son los ejemplos con `[]`.

In [50]:
def proporcion_de_clases(corpus):
    """% de cada clase común dentro de cada split (una fila multietiqueta cuenta en cada clase)."""
    clases = corpus["etiquetas"].map(lambda claves: claves if claves else ["sin_falacia"])
    por_clase = pd.DataFrame({"clase": clases, "split": corpus["split"]}).explode("clase", ignore_index=True)
    tabla = pd.crosstab(por_clase["clase"], por_clase["split"], normalize="columns")
    return (tabla[ORDEN_SPLITS] * 100).round(1)


proporcion_de_clases(proto_ejemplos)

split,train,dev,test
clase,,,
ad_hominem,7.6,7.8,7.8
ad_populum,5.5,5.7,5.5
apelacion_emocion,4.0,4.1,4.1
deduccion_invalida,4.2,4.3,4.3
distraccion,4.0,4.3,4.1
falacia_credibilidad,3.5,3.5,3.7
falsa_causalidad,5.5,5.5,5.7
falso_dilema,3.5,3.5,3.7
generalizacion_apresurada,11.7,12.3,12.1


In [51]:
proporcion_de_clases(espont)

split,train,dev,test
clase,,,
ad_hominem,11.1,10.6,12.8
ad_populum,3.3,3.7,5.3
apelacion_emocion,2.4,1.8,0.8
deduccion_invalida,2.2,2.6,2.3
falacia_credibilidad,9.7,12.1,9.8
falso_dilema,9.8,7.0,13.2
generalizacion_apresurada,2.5,3.3,2.3
hombre_de_paja,8.1,10.6,6.8
sin_falacia,50.7,48.4,47.0


In [52]:
proporcion_de_clases(political)

split,train,dev,test
clase,,,
ad_hominem,23.2,21.2,19.5
ad_populum,2.3,4.8,1.7
apelacion_emocion,14.6,14.0,15.1
deduccion_invalida,3.5,2.7,3.8
distraccion,0.9,2.4,2.6
falacia_credibilidad,4.9,2.7,4.1
falsa_causalidad,15.7,15.7,14.5
falso_dilema,10.1,8.2,6.4
generalizacion_apresurada,2.6,2.4,5.2


Resultado de los splits:
- Los tamaños quedan cerca del 70/15/15. Political tiene algo más de test (17,6 %) porque se reparte por debates enteros.
- Ningún grupo ni ningún texto está en dos splits.
- **Prototípicas**: la proporción de cada clase es casi igual en los tres splits (se repartió manteniéndola).
- **Espontáneas**: las proporciones varían algo más, porque se reparte por noticias al azar y algunas clases tienen muy pocos ejemplos (*apelación a la emoción* tiene 38 en total, y solo 2 caen en test). Es suficiente para evaluar la detección (hay ~50 % sin falacia en los tres splits), pero las métricas por clase de las espontáneas serán poco fiables en las clases pequeñas.
- **Political**: las diferencias entre splits son mayores porque el reparto es por fecha y cada debate tiene su propio estilo (en 2023 hay más *hombre de paja* y menos *falso dilema*). Es el precio de evaluar con debates nuevos, que es el escenario realista.

## 11. Limpieza del texto y exportación a JSONL

Cada ejemplo se guarda con dos versiones del texto:
- `texto_original`: tal cual viene en el corpus, sin tocar.
- `texto`: con una **limpieza mínima**. Es el que se procesa después (notebook 03).

La limpieza solo arregla problemas de formato, sin cambiar palabras:
1. Quitar las marcas de cambio de orador de la transcripción automática (`[CHANGE TO SPEAKER_05]`), que aparecen en algunos textos y contextos de Political.
2. Dejar un solo espacio entre palabras: elimina los dobles espacios de Political y los saltos de línea (`\r\n`) de las espontáneas.
3. Solo en las prototípicas: quitar las comillas que envuelven el texto entero (*"\"O me amas, o me odias\""* → *O me amas, o me odias*). Muchas falacias vienen entrecomilladas y otras no; no aporta nada y así el mismo texto queda igual escrito de las dos formas.

No se quitan tildes, mayúsculas ni signos de puntuación: eso lo decide el preprocesado (notebook 03).

In [53]:
import json

MARCA_DE_ORADOR = r"\[CHANGE TO [^\]]*\]"


def limpiar_texto(texto):
    """Quita las marcas de orador y deja un solo espacio entre palabras."""
    texto = re.sub(MARCA_DE_ORADOR, " ", texto)
    return " ".join(texto.split())  # split() sin argumentos parte por espacios, tabuladores y saltos de línea


def quitar_comillas_externas(texto):
    """Quita las comillas que envuelven el texto entero: '"Hola".' → 'Hola.'
    Solo si son las dos únicas comillas del texto (si hay más, son citas dentro del texto)."""
    entrecomillado = re.fullmatch(r'"([^"]*)"([.!?]?)', texto)
    if entrecomillado:
        return entrecomillado.group(1).strip() + entrecomillado.group(2)
    return texto


# Ejemplos
for prueba in ['"O me amas, o me odias".',
               'Mamá: "¿Has terminado los deberes?" Hijo: "Oye..."',
               "la política  arrogante [CHANGE TO SPEAKER_02]\r\nY esa"]:
    print(repr(prueba), "→", repr(quitar_comillas_externas(limpiar_texto(prueba))))

'"O me amas, o me odias".' → 'O me amas, o me odias.'
'Mamá: "¿Has terminado los deberes?" Hijo: "Oye..."' → 'Mamá: "¿Has terminado los deberes?" Hijo: "Oye..."'
'la política  arrogante [CHANGE TO SPEAKER_02]\r\nY esa' → 'la política arrogante Y esa'


Se construye cada tabla con los campos y el orden del esquema pactado con el grupo. Algunas notas:
- Los IDs de las espontáneas y de Political llevan el número de fila del CSV original (`fes_0042`, `fpo_0042`); los números de las filas quitadas como duplicadas quedan sin usar.
- En Political, el partido `UNKNOWN` se guarda como `null`, igual que los oradores vacíos. Los contextos también se limpian (tienen muchas marcas de orador).

In [54]:
def id_con_numero(prefijo, fila):
    return f"{prefijo}_{fila:04d}"


salida_proto = pd.DataFrame({
    "id": proto_ejemplos["id"],
    "split": proto_ejemplos["split"],
    "texto_original": proto_ejemplos["texto_original"],
    "texto": proto_ejemplos["texto_original"].map(limpiar_texto).map(quitar_comillas_externas),
    "etiquetas_original": proto_ejemplos["etiquetas_original"],
    "etiquetas": proto_ejemplos["etiquetas"],
})

salida_espont = pd.DataFrame({
    "id": [id_con_numero("fes", fila) for fila in espont.index],
    "split": espont["split"],
    "titular_noticia": espont["news_title"],
    "resumen_noticia": espont["news_summary"],
    "enlace_noticia": espont["news_link"],
    "texto_original": espont["text"],
    "texto": espont["text"].map(limpiar_texto),
    "etiquetas_original": espont["etiquetas_original"],
    "etiquetas": espont["etiquetas"],
})

salida_political = pd.DataFrame({
    "id": [id_con_numero("fpo", fila) for fila in political.index],
    "split": political["split"],
    "fecha": political["document"],
    "orador_nombre": political["speaker_name"],
    "partido": political["speaker_party"].replace("UNKNOWN", None),
    "contexto_previo": political["prev_context"].map(limpiar_texto, na_action="ignore"),
    "contexto_posterior": political["foll_context"].map(limpiar_texto, na_action="ignore"),
    "texto_original": political["text"],
    "texto": political["text"].map(limpiar_texto),
    "etiquetas_original": political["etiquetas_original"],
    "etiquetas": political["etiquetas"],
})

salida_political.head(3)

,id,split,fecha,orador_nombre,partido,contexto_previo,contexto_posterior,texto_original,texto,etiquetas_original,etiquetas
0,fpo_0000,train,1993-05-24,José María Aznar,PP,"Pueden ser también los jóvenes hoy desorientados, desanimados porque no encuentran un empleo, porque el precio de la vivienda es muy alto o simplemente porque no tienen la motivación para abrirse ...","Pero si esas son las causas, yo quiero apostar en este momento por las soluciones.","Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista Y esa política, que ha sido una política arrogante, es la que ha gene...","Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista Y esa política, que ha sido una política arrogante, es la que ha gener...",[False Cause],[falsa_causalidad]
1,fpo_0001,train,1993-05-24,José María Aznar,PP,"Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista Y esa política, que ha sido una política arrogante, es la que ha gener...","Un gobierno eficaz, compuesto por personas competentes.","Porque efectivamente hoy los españoles tenemos ante nosotros la oportunidad de recuperar esperanza, de recuperar ilusión, de recuperar capacidad de trabajo. Y es por lo que creo que ha llegado la...","Porque efectivamente hoy los españoles tenemos ante nosotros la oportunidad de recuperar esperanza, de recuperar ilusión, de recuperar capacidad de trabajo. Y es por lo que creo que ha llegado la ...",[Appeal to Emotion],[apelacion_emocion]
2,fpo_0002,train,1993-05-24,José María Aznar,PP,"Yo quiero recordarle, en definitiva, que usted desde el comienzo tuvo un magnífico caudal de entusiasmo ciudadano. Jamás un gobernante ha tenido, democrático, ha tenido el apoyo que usted ha tenid...","Por eso hace falta el cambio ahora mismo. El problema sigue siendo creer o no creer en el país, sin duda alguna.","Me permitirá decirle que con el historial de promesas incumplidas que usted tiene encima, de los españoles para volver a tener la sensibilidad necesaria para acercarse a los problemas de los ciu...","Me permitirá decirle que con el historial de promesas incumplidas que usted tiene encima, de los españoles para volver a tener la sensibilidad necesaria para acercarse a los problemas de los ciuda...",[Ad Hominem],[ad_hominem]


In [55]:
# Cómo ha quedado la limpieza en unos ejemplos de cada corpus
cambiados_proto = salida_proto[salida_proto["texto"] != salida_proto["texto_original"]]
print("Prototípicas con el texto cambiado:", len(cambiados_proto))
print("Espontáneas con el texto cambiado:", (salida_espont["texto"] != salida_espont["texto_original"]).sum())
print("Political con el texto cambiado:", (salida_political["texto"] != salida_political["texto_original"]).sum())
cambiados_proto[["texto_original", "texto"]].head(4)

Prototípicas con el texto cambiado: 305
Espontáneas con el texto cambiado: 182
Political con el texto cambiado: 1198


,texto_original,texto
0,"""A Annie le debe gustar Starbucks porque a todas las chicas blancas les gusta Starbucks"".",A Annie le debe gustar Starbucks porque a todas las chicas blancas les gusta Starbucks.
4,"""Los dos cursos que tomé en la UF no fueron muy interesantes. No creo que sea una buena universidad"".",Los dos cursos que tomé en la UF no fueron muy interesantes. No creo que sea una buena universidad.
46,"""¡Esa nueva casa se ve muy bien! Comprémosla"".",¡Esa nueva casa se ve muy bien! Comprémosla.
52,"""Los dos cursos que hice en Harvard no fueron muy interesantes. No creo que sea una buena universidad"".",Los dos cursos que hice en Harvard no fueron muy interesantes. No creo que sea una buena universidad.


Se guardan en `data/processed/`, que sí se sube a GitHub (estos ficheros no se pueden regenerar sin ejecutar todo el notebook, y el resto del grupo los necesita). Formato JSONL: una línea por ejemplo, en UTF-8 y con las tildes tal cual (`ensure_ascii=False`). Los valores que faltan se guardan como `null`.

In [56]:
CARPETA_SALIDA = Path("../data/processed")


def guardar_jsonl(tabla, nombre_fichero):
    """Guarda la tabla en JSONL (una línea por fila). Los valores vacíos (NaN) se guardan como null."""
    ruta = CARPETA_SALIDA / nombre_fichero
    registros = tabla.astype(object).where(tabla.notna(), None).to_dict(orient="records")
    with open(ruta, "w", encoding="utf-8") as fichero:
        for registro in registros:
            fichero.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print(f"Guardado {ruta.name}: {len(registros)} ejemplos")


guardar_jsonl(salida_proto, "fallacyes_proto_limpio.jsonl")
guardar_jsonl(salida_espont, "fallacyes_espont_limpio.jsonl")
guardar_jsonl(salida_political, "fallacyes_political_limpio.jsonl")

Guardado fallacyes_proto_limpio.jsonl: 3319 ejemplos
Guardado fallacyes_espont_limpio.jsonl: 1840 ejemplos
Guardado fallacyes_political_limpio.jsonl: 1951 ejemplos


### Comprobación: se vuelven a leer con pandas

Se leen los tres ficheros como lo hará el resto del grupo y se comprueba que no se ha perdido ni cambiado nada.

In [57]:
for nombre_fichero, tabla in [
    ("fallacyes_proto_limpio.jsonl", salida_proto),
    ("fallacyes_espont_limpio.jsonl", salida_espont),
    ("fallacyes_political_limpio.jsonl", salida_political),
]:
    leido = pd.read_json(CARPETA_SALIDA / nombre_fichero, lines=True)
    print(nombre_fichero)
    print("   filas:", len(leido), "| iguales que al guardar:", len(leido) == len(tabla))
    print("   columnas en el mismo orden:", list(leido.columns) == list(tabla.columns))
    print("   IDs repetidos:", leido["id"].duplicated().sum())
    print("   textos vacíos:", (leido["texto"].str.strip() == "").sum())
    print("   etiquetas que no son lista:", (~leido["etiquetas"].map(lambda e: isinstance(e, list))).sum())
    print("   quedan marcas de orador o dobles espacios:",
          leido["texto"].str.contains(MARCA_DE_ORADOR).sum() + leido["texto"].str.contains("  ").sum())
    print("   splits:", leido["split"].value_counts().to_dict())

fallacyes_proto_limpio.jsonl
   filas: 3319 | iguales que al guardar: True
   columnas en el mismo orden: True
   IDs repetidos: 0
   textos vacíos: 0
   etiquetas que no son lista: 0
   quedan marcas de orador o dobles espacios: 0
   splits: {'train': 2341, 'test': 489, 'dev': 489}
fallacyes_espont_limpio.jsonl
   filas: 1840 | iguales que al guardar: True
   columnas en el mismo orden: True
   IDs repetidos: 0
   textos vacíos: 0
   etiquetas que no son lista: 0
   quedan marcas de orador o dobles espacios: 0
   splits: {'train': 1301, 'dev': 273, 'test': 266}
fallacyes_political_limpio.jsonl
   filas: 1951 | iguales que al guardar: True
   columnas en el mismo orden: True
   IDs repetidos: 0
   textos vacíos: 0
   etiquetas que no son lista: 0
   quedan marcas de orador o dobles espacios: 0
   splits: {'train': 1316, 'test': 344, 'dev': 291}


In [58]:
# Así se ve un ejemplo de cada fichero
with open(CARPETA_SALIDA / "fallacyes_espont_limpio.jsonl", encoding="utf-8") as fichero:
    print(fichero.readline())

{"id": "fes_0000", "split": "train", "titular_noticia": "Cómo los milmillonarios provocaron la caída del Imperio romano", "resumen_noticia": "La causa del hundimiento del Imperio romano de Occidente fue la avaricia de las élites, las luchas intestinas que entablaron entre sí para alcanzar el poder y su creciente desapego del Estado, afirmado en la creencia equivocada de que podían prescindir de él y de las garantías que brindaba debido al respaldo que les proporcionaban sus propias fortunas. En el 425 d. C., las rentas de la vieja aristocracia romana eran altísimas. Una sola familia obtenía 4.000 libras de oro al año, la mitad del presupuesto militar romano.", "enlace_noticia": "https://www.larazon.es/cultura/historia/20221009/w57unlibcneo7bjqbf4tjwn6vq.html?outputType=amp", "texto_original": "Para mi lo que diga este tipejo tiene cero validez.", "texto": "Para mi lo que diga este tipejo tiene cero validez.", "etiquetas_original": ["ad hominem"], "etiquetas": ["ad_hominem"]}

